# Loan Default Risk & Pricing — LendingClub 2007–2018

| Part | Sections | Owner | Viva focus |
|---|---|---|---|
| **A** | 1–8 | Student 1 | Data, leakage policy, EDA, features, baselines |
| **B** | 9–12 | Student 2 | Bias–variance, Random Forest, XGBoost tuning, feature selection |
| **C** | 13–17 | Student 3 | Model comparison, calibration, evaluation, pricing, deployment |

In [1]:
import os, gc, json, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, validation_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, RocCurveDisplay, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
N_ITER  = int(os.environ.get('LC_NITER', 18))
N_BOOT  = int(os.environ.get('LC_NBOOT', 500))
plt.rcParams['figure.dpi'] = 110

## 1. Problem

A lender commits money before knowing whether it comes back. We predict the **probability of default** using only what is known at application time.

Two errors, unequal costs: **approving a defaulter** loses most of the principal; **rejecting a good borrower** loses only the profit on that loan. We assume the first mistake costs **5×** more — that asymmetry drives every downstream decision on threshold and pricing.

Live app: **http://eab-credit.preview.james-benjamin.com/**

## 2. Data & Leakage Policy

| Group | Examples | Role |
|---|---|---|
| Application-time | loan amount, income, DTI, FICO, credit bureau counts | **Model inputs** |
| LC's own pricing | `int_rate`, `grade`, `sub_grade` | **Benchmark only** — never a model input |
| Outcome | `loan_status` | Builds the target |
| Post-origination | payments received, recoveries | **LGD measurement only** |

Using LC's pricing as a model input would be circular: we would be predicting risk from risk.

In [2]:
import kagglehub
DATA_DIR = kagglehub.dataset_download('wordsforthewise/lending-club')
DATA_PATH = str(list(pathlib.Path(DATA_DIR).rglob('accepted_2007_to_2018Q4.csv*'))[0])

ALL_COLS = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

NUMERIC_CANDS = [
    'loan_amnt','annual_inc','dti','delinq_2yrs','inq_last_6mths','open_acc','pub_rec',
    'revol_bal','revol_util','total_acc','collections_12_mths_ex_med','acc_now_delinq',
    'tot_coll_amt','tot_cur_bal','total_rev_hi_lim','acc_open_past_24mths','avg_cur_bal',
    'bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt',
    'mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc',
    'mths_since_recent_inq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
    'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts',
    'num_rev_tl_bal_gt_0','num_sats','num_tl_120dpd_2m','num_tl_30dpd',
    'num_tl_90g_dpd_24m','pct_tl_nvr_dlq','percent_bc_gt_75','pub_rec_bankruptcies',
    'tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit',
    'total_il_high_credit_limit','fico_range_low','fico_range_high',
]
CAT_CANDS   = ['term','emp_length','home_ownership','verification_status','purpose','application_type']
DATE_COLS   = ['issue_d','earliest_cr_line']
BENCHMARK   = ['int_rate','grade','sub_grade']
OUTCOME     = ['funded_amnt','total_rec_prncp','recoveries']

keep = lambda cols: [c for c in cols if c in ALL_COLS]
USECOLS = sorted(set(
    keep(NUMERIC_CANDS) + keep(CAT_CANDS) + keep(DATE_COLS)
    + ['loan_status'] + keep(BENCHMARK) + keep(OUTCOME)
))

parts = []
for i, chunk in enumerate(pd.read_csv(DATA_PATH, usecols=USECOLS, chunksize=200_000, low_memory=False)):
    chunk = chunk[chunk['loan_status'].isin(['Fully Paid','Charged Off'])]
    if len(chunk):
        parts.append(chunk.sample(frac=0.08, random_state=RANDOM_STATE + i))
df = pd.concat(parts, ignore_index=True)
if len(df) > 150_000:
    df = df.sample(150_000, random_state=RANDOM_STATE)
del parts; gc.collect()

print(f'Sample: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Default rate: {(df["loan_status"]=="Charged Off").mean():.1%}')

Sample: 107,625 rows, 64 columns
Default rate: 20.0%


## 3. Cleaning & Target

Rules use fixed business limits, not data percentiles — no information leaks from test into train.

In [3]:
DF = df.copy()
DF['default'] = (DF['loan_status'] == 'Charged Off').astype(int)

# Type fixes
for c in keep(DATE_COLS):
    DF[c] = pd.to_datetime(DF[c], format='%b-%Y', errors='coerce')
if 'revol_util' in DF and DF['revol_util'].dtype == object:
    DF['revol_util'] = pd.to_numeric(DF['revol_util'].str.rstrip('%'), errors='coerce')
DF['term'] = DF['term'].astype(str).str.strip()
DF['fico_avg'] = DF[['fico_range_low','fico_range_high']].mean(axis=1)

# Business caps
before = len(DF)
DF = DF[DF['annual_inc'] > 0].copy()
DF['dti']       = DF['dti'].clip(0, 100)
DF['revol_util'] = DF['revol_util'].clip(0, 150)
DF['annual_inc'] = DF['annual_inc'].clip(upper=1_000_000)
DF = DF.replace([np.inf, -np.inf], np.nan).dropna(subset=['issue_d'])
DF = DF.sort_values('issue_d').reset_index(drop=True)

print(f'Removed {before - len(DF):,} rows | {len(DF):,} remain | default rate {DF["default"].mean():.1%}')

Removed 31 rows | 107,594 remain | default rate 20.0%
